# Task 005a — Train churn LightGBM + SHAP

Trains the LightGBM binary churn classifier on the 70% training split, evaluates on the 20% test split, and produces per-test-client SHAP top-5 explanations.

**Outputs**
- `data/models/churn_model.pkl` — fitted classifier
- `data/models/churn_explainer.pkl` — SHAP TreeExplainer
- `data/models/churn_metrics.json` — test-set metrics
- `data/processed/churn_test_shap_top5.parquet` — per-test-client top-5 SHAP contributions

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

In [2]:
import json
import pickle
import warnings

import numpy as np
import pandas as pd
import lightgbm as lgb
import shap
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix
from sklearn.model_selection import train_test_split

import config

PROC = config.PROCESSED_DATA_DIR
MODELS = config.MODELS_DIR
MODELS.mkdir(parents=True, exist_ok=True)

## 1. Load features, labels, splits, feature column list

In [3]:
features = pd.read_parquet(PROC / 'features_churn.parquet')
labels   = pd.read_parquet(PROC / 'labels.parquet')

with open(PROC / 'split_train.json') as f: train_ids = json.load(f)
with open(PROC / 'split_test.json')  as f: test_ids  = json.load(f)

with open(MODELS / 'churn_feature_columns.json') as f: feature_cols = json.load(f)

print(f'features shape : {features.shape}')
print(f'labels shape   : {labels.shape}')
print(f'train ids      : {len(train_ids):,}')
print(f'test ids       : {len(test_ids):,}')
print(f'feature cols   : {len(feature_cols)}')

features shape : (4614, 83)
labels shape   : (4152, 5)
train ids      : 3,126
test ids       : 1,026
feature cols   : 82


## 2. Build X, y for train and test

Set features index on client ID; join labels. Convert `object` columns on the training matrix to pandas `category` dtype using a single categorical definition per column, then reapply that exact dtype to validation and test matrices so LightGBM sees consistent categorical codes across all splits.

In [4]:
features = features.set_index('IDENTIFIKATOR_KLIJENTA')
label_map = labels.set_index('IDENTIFIKATOR_KLIJENTA')['churned']

X_train_full = features.loc[train_ids, feature_cols].copy()
X_test       = features.loc[test_ids,  feature_cols].copy()
y_train_full = label_map.loc[train_ids].astype(int)
y_test       = label_map.loc[test_ids].astype(int)

assert not y_train_full.isna().any(), 'NaN in train labels!'
assert not y_test.isna().any(),       'NaN in test labels!'

# Freeze categorical dtypes on the training matrix and apply to test
obj_cols = X_train_full.select_dtypes(include=['object']).columns.tolist()
cat_dtypes = {}
for col in obj_cols:
    X_train_full[col] = X_train_full[col].astype('category')
    cat_dtypes[col] = X_train_full[col].dtype
for col, dt in cat_dtypes.items():
    X_test[col] = X_test[col].astype(dt)

print(f'X_train_full : {X_train_full.shape}   churn rate: {y_train_full.mean():.4f}')
print(f'X_test       : {X_test.shape}   churn rate: {y_test.mean():.4f}')
print(f'Categorical cols ({len(obj_cols)}): {obj_cols}')

X_train_full : (3126, 82)   churn rate: 0.1500
X_test       : (1026, 82)   churn rate: 0.1287
Categorical cols (12): ['gender', 'employment_status', 'employment_type', 'position', 'education', 'occupation', 'employer_category', 'employer_type', 'housing_type', 'ownership_type', 'marital_status', 'acquisition_channel']


## 3. Validation fold for early stopping

Carve an 80/20 stratified split off the training data. The validation fold is the early-stopping signal only — it never leaks into test metrics.

In [5]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=0.20, stratify=y_train_full, random_state=42,
)

# Reapply the frozen categorical dtype to the validation fold
for col, dt in cat_dtypes.items():
    X_tr[col]  = X_tr[col].astype(dt)
    X_val[col] = X_val[col].astype(dt)

print(f'X_tr  : {X_tr.shape}   churn rate: {y_tr.mean():.4f}')
print(f'X_val : {X_val.shape}   churn rate: {y_val.mean():.4f}')

X_tr  : (2500, 82)   churn rate: 0.1500
X_val : (626, 82)   churn rate: 0.1502


## 4. Train LightGBM

Hyperparameters come from `config.CHURN_PARAMS`. Early stopping at 30 rounds against the validation fold.

In [6]:
model = lgb.LGBMClassifier(**config.CHURN_PARAMS, random_state=42)
model.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(30), lgb.log_evaluation(50)],
)

print(f'best_iteration : {model.best_iteration_}')
print(f'n_features_in_ : {model.n_features_in_}')

Training until validation scores don't improve for 30 rounds


[50]	valid_0's binary_logloss: 0.210503


[100]	valid_0's binary_logloss: 0.172226


[150]	valid_0's binary_logloss: 0.15401


[200]	valid_0's binary_logloss: 0.147438


[250]	valid_0's binary_logloss: 0.148082
Early stopping, best iteration is:
[229]	valid_0's binary_logloss: 0.146534
best_iteration : 229
n_features_in_ : 82


## 5. Evaluate on test set

In [7]:
proba_test = model.predict_proba(X_test)[:, 1]
pred_test  = (proba_test >= 0.5).astype(int)

auc       = roc_auc_score(y_test, proba_test)
avg_prec  = average_precision_score(y_test, proba_test)

# Precision at top 10%
n_test     = len(y_test)
k_top10    = max(1, int(round(n_test * 0.10)))
order      = np.argsort(-proba_test)
top10_idx  = order[:k_top10]
precision_at_top10 = float(y_test.values[top10_idx].mean())

# Lift at decile 1 (top 10%)
base_rate      = float(y_test.mean())
lift_decile1   = precision_at_top10 / base_rate if base_rate > 0 else float('nan')

cm = confusion_matrix(y_test, pred_test).tolist()

print(f'AUC-ROC            : {auc:.4f}')
print(f'Average precision  : {avg_prec:.4f}')
print(f'Precision @ top 10%: {precision_at_top10:.4f}  (k={k_top10})')
print(f'Lift @ decile 1    : {lift_decile1:.2f}x  (base rate={base_rate:.4f})')
print(f'Confusion matrix (threshold=0.5):')
print(f'  TN={cm[0][0]}  FP={cm[0][1]}')
print(f'  FN={cm[1][0]}  TP={cm[1][1]}')

if auc < 0.85:
    warnings.warn(f'AUC {auc:.4f} below target 0.85', stacklevel=2)
if lift_decile1 < 3.5:
    warnings.warn(f'Lift@D1 {lift_decile1:.2f} below target 3.5', stacklevel=2)

AUC-ROC            : 0.9576
Average precision  : 0.8423
Precision @ top 10%: 0.8058  (k=103)
Lift @ decile 1    : 6.26x  (base rate=0.1287)
Confusion matrix (threshold=0.5):
  TN=847  FP=47
  FN=27  TP=105


## 6. SHAP TreeExplainer

For each test client, extract the top-5 features by absolute SHAP impact on the class-1 (churn) probability. Handles both return shapes of `TreeExplainer.shap_values`: older list-of-two-arrays and newer single 2D array.

In [8]:
explainer = shap.TreeExplainer(model)
sv_raw = explainer.shap_values(X_test)

# Normalize to (n_test, n_features) array for the positive (churn) class
if isinstance(sv_raw, list):
    shap_values_test = np.asarray(sv_raw[1])
else:
    shap_values_test = np.asarray(sv_raw)
    if shap_values_test.ndim == 3:
        # shape (n, n_feat, n_classes) — take class 1
        shap_values_test = shap_values_test[:, :, 1]

assert shap_values_test.shape == X_test.shape, (
    f'SHAP shape {shap_values_test.shape} != X_test shape {X_test.shape}'
)
print(f'SHAP values shape: {shap_values_test.shape}')

SHAP values shape: (1026, 82)


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/shap/explainers/_tree.py:620: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


In [9]:
# For each test client, grab indices of top-5 features by |shap_impact|
abs_sv   = np.abs(shap_values_test)
top_idx  = np.argsort(-abs_sv, axis=1)[:, :5]

client_ids = X_test.index.to_numpy()
feat_arr   = np.asarray(feature_cols)

rows = []
X_values = X_test.values  # preserves per-column dtypes well enough for casting below
for row_i, cid in enumerate(client_ids):
    for rank, col_i in enumerate(top_idx[row_i], start=1):
        raw_val = X_values[row_i, col_i]
        # cast categorical / NaN safely to a JSON-friendly scalar
        if pd.isna(raw_val):
            raw_out = None
        elif isinstance(raw_val, (np.floating, np.integer)):
            raw_out = float(raw_val)
        else:
            raw_out = str(raw_val)
        rows.append({
            'client_id':   cid,
            'rank':        rank,
            'feature':     feat_arr[col_i],
            'raw_value':   raw_out,
            'shap_impact': float(shap_values_test[row_i, col_i]),
        })

shap_top5 = pd.DataFrame(rows)
out_shap = PROC / 'churn_test_shap_top5.parquet'
shap_top5.to_parquet(out_shap, index=False)
print(f'Saved: {out_shap}  ({len(shap_top5):,} rows = {len(client_ids):,} clients x 5)')
shap_top5.head(10)

Saved: /Users/Max/Desktop/Fintech-hackathon/data/processed/churn_test_shap_top5.parquet  (5,130 rows = 1,026 clients x 5)


,client_id,rank,feature,raw_value,shap_impact
0,VH595J0YEXTR,1,days_since_last_tx,522,3.632254
1,VH595J0YEXTR,2,tx_count_ratio_30d_180d,None,1.135662
2,VH595J0YEXTR,3,receives_primary_income_at_bank,0,0.682118
3,VH595J0YEXTR,4,tx_count_total,12,0.427305
4,VH595J0YEXTR,5,n_products_closed_last_12m,1,0.408501
5,9ZKRZNR5435B,1,receives_primary_income_at_bank,1,-0.715326
6,9ZKRZNR5435B,2,total_balance_avg,154028.15263157897,0.633384
7,9ZKRZNR5435B,3,tx_amount_trend_slope,-109.42712280701753,0.625691
8,9ZKRZNR5435B,4,days_since_last_tx,6,-0.577118
9,9ZKRZNR5435B,5,total_balance_end,59462.79,-0.424085


## 7. Save artifacts

In [10]:
out_model   = MODELS / 'churn_model.pkl'
out_expl    = MODELS / 'churn_explainer.pkl'
out_metrics = MODELS / 'churn_metrics.json'

with open(out_model, 'wb') as f:
    pickle.dump(model, f)
print(f'Saved: {out_model}')

with open(out_expl, 'wb') as f:
    pickle.dump(explainer, f)
print(f'Saved: {out_expl}')

metrics_out = {
    'auc_roc':            float(auc),
    'average_precision':  float(avg_prec),
    'precision_at_top10': float(precision_at_top10),
    'lift_decile1':       float(lift_decile1),
    'base_rate':          float(base_rate),
    'confusion_matrix':   cm,
    'best_iteration':     int(model.best_iteration_) if model.best_iteration_ is not None else None,
    'n_features':         int(model.n_features_in_),
    'feature_cols_count': len(feature_cols),
    'n_train':            int(len(X_tr)),
    'n_val':              int(len(X_val)),
    'n_test':             int(len(X_test)),
}
with open(out_metrics, 'w') as f:
    json.dump(metrics_out, f, indent=2)
print(f'Saved: {out_metrics}')

Saved: /Users/Max/Desktop/Fintech-hackathon/data/models/churn_model.pkl
Saved: /Users/Max/Desktop/Fintech-hackathon/data/models/churn_explainer.pkl
Saved: /Users/Max/Desktop/Fintech-hackathon/data/models/churn_metrics.json


## 8. Sanity checks

In [11]:
print('=' * 60)
print('ACCEPTANCE CRITERIA CHECKS')
print('=' * 60)

# 1. Model fitted
assert hasattr(model, 'booster_') and model.booster_ is not None, 'Model not fitted!'
print('[PASS] 1. Model is fitted (model.booster_ present)')

# 2. Feature count matches
assert model.n_features_in_ == len(feature_cols), (
    f'model.n_features_in_ {model.n_features_in_} != len(feature_cols) {len(feature_cols)}'
)
print(f'[PASS] 2. n_features_in_ ({model.n_features_in_}) == len(feature_cols) ({len(feature_cols)})')

# 3. Metrics file complete
with open(out_metrics) as f: loaded = json.load(f)
required_keys = {
    'auc_roc', 'average_precision', 'precision_at_top10', 'lift_decile1',
    'base_rate', 'confusion_matrix', 'best_iteration',
    'n_features', 'feature_cols_count', 'n_train', 'n_val', 'n_test',
}
missing = required_keys - set(loaded.keys())
assert not missing, f'metrics file missing keys: {missing}'
print(f'[PASS] 3. Metrics file has all {len(required_keys)} required keys')

# 4. SHAP top-5 parquet shape
shap_check = pd.read_parquet(out_shap)
assert len(shap_check) == len(test_ids) * 5, (
    f'SHAP rows {len(shap_check)} != {len(test_ids) * 5}'
)
assert set(shap_check.columns) == {'client_id', 'rank', 'feature', 'raw_value', 'shap_impact'}
assert shap_check['rank'].between(1, 5).all()
print(f'[PASS] 4. churn_test_shap_top5.parquet: {len(shap_check):,} rows, correct columns, ranks 1-5')

# 5. Model and explainer pickles load back
with open(out_model, 'rb') as f: m2 = pickle.load(f)
with open(out_expl, 'rb') as f:  e2 = pickle.load(f)
_ = m2.predict_proba(X_test.head(3))
print('[PASS] 5. Pickled model and explainer load and predict on a 3-row sample')

print()
print('ALL CHECKS PASSED.')

ACCEPTANCE CRITERIA CHECKS
[PASS] 1. Model is fitted (model.booster_ present)
[PASS] 2. n_features_in_ (82) == len(feature_cols) (82)
[PASS] 3. Metrics file has all 12 required keys
[PASS] 4. churn_test_shap_top5.parquet: 5,130 rows, correct columns, ranks 1-5


[PASS] 5. Pickled model and explainer load and predict on a 3-row sample

ALL CHECKS PASSED.
